In [1]:
import os

os.environ["HF_HOME"] = "C:/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "C:/hf_cache/datasets"
os.environ["HF_HUB_CACHE"] = "C:/hf_cache/hub"

import json

with open("../eval_responses.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

instructions = eval_data["instructions"]
references = eval_data["references"]
base_responses = eval_data["base_responses"]
finetuned_responses = eval_data["finetuned_responses"]

print("Reloaded", len(instructions), "examples")

Reloaded 50 examples


In [2]:
import torch
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

JUDGE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME, trust_remote_code=True)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

judge_pipe = pipeline(
    "text-generation",
    model=judge_model,
    tokenizer=judge_tokenizer,
    max_new_tokens=512,
    temperature=0.01,
    do_sample=False,
    return_full_text=False,
)

llm_judge = HuggingFacePipeline(pipeline=judge_pipe)

print("Judge model loaded and wrapped for Ragas")

C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Judge model loaded and wrapped for Ragas


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embeddings model loaded")

C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\hf_cache\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Li

Embeddings model loaded


In [4]:
from datasets import Dataset

base_eval_data = {
    "question": instructions,
    "answer": base_responses,
    "contexts": [[ref] for ref in references],  # reference wrapped as single-item context list
    "ground_truth": references,
}

finetuned_eval_data = {
    "question": instructions,
    "answer": finetuned_responses,
    "contexts": [[ref] for ref in references],
    "ground_truth": references,
}

base_ragas_dataset = Dataset.from_dict(base_eval_data)
finetuned_ragas_dataset = Dataset.from_dict(finetuned_eval_data)

print("Base Ragas dataset:", base_ragas_dataset)
print("\nFine-tuned Ragas dataset:", finetuned_ragas_dataset)

Base Ragas dataset: Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 50
})

Fine-tuned Ragas dataset: Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 50
})


In [5]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision

# Small test slice first
base_test_slice = base_ragas_dataset.select(range(5))

test_result = evaluate(
    base_test_slice,
    metrics=[faithfulness, context_precision],
    llm=llm_judge,
    embeddings=embeddings,
)

print(test_result)

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is on

{'faithfulness': nan, 'context_precision': 1.0000}


In [6]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision
from ragas.run_config import RunConfig

# Local models are slower than API judges - give it much more time per call,
# and run requests one at a time since there's no real parallelism benefit
# with a single local GPU model anyway
custom_run_config = RunConfig(
    timeout=300,       # 5 minutes per call instead of the short default
    max_workers=1,     # sequential, avoids queuing/timeout issues on local GPU
)

test_result = evaluate(
    base_test_slice,
    metrics=[faithfulness, context_precision],
    llm=llm_judge,
    embeddings=embeddings,
    run_config=custom_run_config,
)

print(test_result)

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is on

{'faithfulness': 0.5000, 'context_precision': 1.0000}


In [7]:
base_result = evaluate(
    base_ragas_dataset,
    metrics=[faithfulness, context_precision],
    llm=llm_judge,
    embeddings=embeddings,
    run_config=custom_run_config,
)

print("BASE MODEL RESULTS:")
print(base_result)

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is o

BASE MODEL RESULTS:
{'faithfulness': 0.4578, 'context_precision': 0.9000}


In [8]:
import json

base_scores = {
    "faithfulness": base_result["faithfulness"],
    "context_precision": base_result["context_precision"],
}

with open("../base_model_ragas_results.json", "w") as f:
    json.dump(base_scores, f, indent=2)

print("Base model results saved:", base_scores)

Base model results saved: {'faithfulness': 0.4577711640211641, 'context_precision': 0.8999999999099999}


In [9]:
finetuned_result = evaluate(
    finetuned_ragas_dataset,
    metrics=[faithfulness, context_precision],
    llm=llm_judge,
    embeddings=embeddings,
    run_config=custom_run_config,
)

print("FINE-TUNED MODEL RESULTS:")
print(finetuned_result)

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is o

FINE-TUNED MODEL RESULTS:
{'faithfulness': 0.6601, 'context_precision': 0.9000}


In [10]:
import json

finetuned_scores = {
    "faithfulness": finetuned_result["faithfulness"],
    "context_precision": finetuned_result["context_precision"],
}

with open("../finetuned_model_ragas_results.json", "w") as f:
    json.dump(finetuned_scores, f, indent=2)

final_comparison = {
    "base_model": {
        "faithfulness": base_scores["faithfulness"],
        "context_precision": base_scores["context_precision"],
        "hallucination_rate": round(1 - base_scores["faithfulness"], 4),
    },
    "finetuned_model": {
        "faithfulness": finetuned_scores["faithfulness"],
        "context_precision": finetuned_scores["context_precision"],
        "hallucination_rate": round(1 - finetuned_scores["faithfulness"], 4),
    },
}

with open("../final_comparison_results.json", "w") as f:
    json.dump(final_comparison, f, indent=2)

print(json.dumps(final_comparison, indent=2))

{
  "base_model": {
    "faithfulness": 0.4577711640211641,
    "context_precision": 0.8999999999099999,
    "hallucination_rate": 0.5422
  },
  "finetuned_model": {
    "faithfulness": 0.660142857142857,
    "context_precision": 0.8999999999099999,
    "hallucination_rate": 0.3399
  }
}


In [11]:
import mlflow

mlflow.set_tracking_uri("file:///" + os.path.abspath("../mlflow/mlruns").replace("\\", "/"))
mlflow.set_experiment("qwen2.5-1.5b-qlora-customer-support")

with mlflow.start_run(run_name="ragas_evaluation_base_vs_finetuned"):
    mlflow.log_metric("base_faithfulness", final_comparison["base_model"]["faithfulness"])
    mlflow.log_metric("base_context_precision", final_comparison["base_model"]["context_precision"])
    mlflow.log_metric("base_hallucination_rate", final_comparison["base_model"]["hallucination_rate"])
    mlflow.log_metric("finetuned_faithfulness", final_comparison["finetuned_model"]["faithfulness"])
    mlflow.log_metric("finetuned_context_precision", final_comparison["finetuned_model"]["context_precision"])
    mlflow.log_metric("finetuned_hallucination_rate", final_comparison["finetuned_model"]["hallucination_rate"])
    mlflow.log_param("eval_sample_size", 50)
    mlflow.log_param("judge_model", "Qwen/Qwen2.5-1.5B-Instruct (local)")

print("Ragas results logged to MLflow")

Ragas results logged to MLflow
